List frontend dependencies to csv file based on package.json file

In [1]:
import json
import pandas as pd
import requests
import sys

In [2]:
def get_package_info_from_npm_registry(
    package_name: str, 
    package_version: str
) -> dict:
    try:
        return requests.get(f'https://registry.npmjs.org/{package_name}/{package_version}').json()
    except KeyError:
        raise RuntimeError(f'Failed to find package "{package_name}" version "{package_version}" in npm registry')
    

def get_dependency_data(dependency_name: str, version: str, is_dev_deps: bool) -> dict:
    sanitized_version: str = version\
        .replace('^', '')\
        .replace('~', '')\
        .replace('<', '')\
        .replace('>', '')\
        .replace('=', '')
    package_info: dict = get_package_info_from_npm_registry(dependency_name, sanitized_version)
    return {
        'Nazwa zależności': dependency_name,
        'Wersja': f'"{sanitized_version}"',
        'Licencja': package_info['license'],
        'Adres repozytorium': package_info['repository']['url'],
        'Dev dependency': 'tak' if is_dev_deps else 'nie',
    }

In [3]:
with open('../frontend/package.json', 'r') as package_json_file:
    package_json: dict = json.load(package_json_file)
    dependencies: dict[str, str] = package_json['dependencies']
    dev_dependencies: dict[str, str] = package_json['devDependencies']

all_dependencies = {
    **{name: (version, False) for name, version in dependencies.items()},
    **{name: (version, True) for name, version in dev_dependencies.items()}
}

progress: int = 1
all_deps_count: int = len(all_dependencies)

data = []
for dependency_name, (version, is_dev_deps) in all_dependencies.items():
    sys.stdout.write('\r')
    # the exact output you're looking for:
    sys.stdout.write(
        '[' + ('#' * progress) + 
        ('_' * (all_deps_count - progress)) + 
        f'] {round(progress / all_deps_count * 100, 2)}%  ' +
        dependency_name
    )
    sys.stdout.flush()
    progress += 1
    data.append(get_dependency_data(dependency_name, version, is_dev_deps))

df = pd.DataFrame(data)
df

[#############################################################] 100.0%  typescriptergin-organize-importsplate

,Nazwa zależności,Wersja,Licencja,Adres repozytorium,Dev dependency
0,@angular/animations,"""15.2.2""",MIT,git+https://github.com/angular/angular.git,nie
1,@angular/cdk,"""15.2.2""",MIT,git+https://github.com/angular/components.git,nie
2,@angular/common,"""15.2.2""",MIT,git+https://github.com/angular/angular.git,nie
3,@angular/compiler,"""15.2.2""",MIT,git+https://github.com/angular/angular.git,nie
4,@angular/core,"""15.2.2""",MIT,git+https://github.com/angular/angular.git,nie
...,...,...,...,...,...
56,prettier-plugin-organize-imports,"""3.2.2""",MIT,git+https://github.com/simonhaenisch/prettier-...,tak
57,sass,"""1.59.2""",MIT,git+https://github.com/sass/dart-sass.git,tak
58,sass-loader,"""13.2.0""",MIT,git+https://github.com/webpack-contrib/sass-lo...,tak
59,style-loader,"""3.3.1""",MIT,git+https://github.com/webpack-contrib/style-l...,tak


In [61]:
df.to_csv('./dependencies.csv', index=False, sep=';')